# AMEX Enterprise Credit Risk Platform
## Notebook 05 — Model Development
### Phase 1 · Problem Statement 1: Credit Scoring / PD Prediction

CRISP-DM stage: **Modeling**. Sprint 2, Notebook 5 of 18. Depends on Notebooks 01-04 (reads `project_config.json`, `notebook_02_summary.json`, `notebook_04_summary.json`) -- run those first if you have not already. **Re-run Notebook 01 once first if you built it before this version** -- it now writes a `resource_limits` block (95% CPU / 90% RAM ceilings) that this notebook reads; it falls back safely to full-core behavior if that block is missing, but the RAM ceiling won't show as configured until 01 is re-run (fast: it only reads label counts and writes config).

**What this notebook does:** trains and compares **7 candidate models** for predicting customer default (Logistic Regression, Random Forest, Extra Trees, sklearn Histogram Gradient Boosting, XGBoost, LightGBM, CatBoost) on the engineered feature store Notebook 04 produced, then selects a champion and generates the true unlabeled-test submission file.

- **Official AMEX competition metric implemented from scratch** -- `0.5 x Normalized Gini + 0.5 x Top-4%-Capture-Rate`, vectorized in numpy, matching the official host-published reference formula (cross-verified against the canonical pandas implementation during this notebook's own build-time testing).
- **Two-layer evaluation**: 5-fold stratified cross-validation *within* Notebook 02's labeled train split (variance estimate), plus a genuinely unbiased evaluation on Notebook 02's held-out test split -- customers this notebook's models never see during training or preprocessing-fit.
- **Champion selection** is by holdout AMEX metric (the actual competition ranking criterion), not by AUC alone.
- Every model, every metric, every chart below is produced live by this cell during this run -- **zero-fabrication rule**, same as Notebooks 01-04.

**Memory-safe by design.** On the real ~925K-row unlabeled test set with ~1,800 engineered feature columns, preprocessing is done entirely in Polars (vectorized, columnar) rather than pandas, every array is float32 (not float64), source Polars frames are freed the moment their numpy array is extracted, and the true test set is scored in bounded chunks -- not loaded and predicted as one giant block. An earlier version of this notebook used pandas' `.replace([inf, -inf], nan)` on the full-width unlabeled test frame, which builds a full-frame boolean mask per replaced value and raised `MemoryError` on a real machine; that pattern has been removed.

**Resource ceilings honored, honestly scoped.** Every model's thread/process count is capped at 95% of detected logical cores (`warp_thread_count` from Notebook 01's config). A 90% RAM ceiling is read and used to size this notebook's memory-aware choices (float32, chunking). CPU **clock speed** ("95% of 5GHz") is *not* something a Python process can set -- boost clocks are controlled by the OS power plan and CPU firmware, not application code -- so this notebook does not claim to control it; only thread count and RAM usage are actually enforced.

**Honest scope note:** categorical columns are label-encoded (fit on train only) and treated as ordinal integers by every model, including the tree-based ones -- a standard, defensible simplification for a first model pass, not native categorical handling. Preprocessing (median imputation, label-encoding, scaling) is fit once on the full train split rather than per cross-validation fold; because none of these statistics derive from the target, the resulting optimism bias is negligible.

**CatBoost is optional.** If it isn't installed, this cell detects that, skips it with a clear message, and still runs the other 6 models -- it does not crash. Install it with `pip install catboost` to include it.

**Run the single code cell below, once.** Idempotent -- every output file (models, comparison tables, submission file) is written to a fixed path and overwritten in place on every re-run. This is the most compute- and memory-intensive notebook so far.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOKS 01-04
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01-04")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB04_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_04_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB04_SUMMARY_PATH, "run 04_feature_engineering.ipynb first"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix} -- this notebook reads its outputs.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB04_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB04_SUMMARY = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]

# --- Resource ceilings (Notebook 01's resource_limits block): 95% of logical
#     cores for every thread/process-count parameter below, and 90% of
#     detected total RAM as a planning ceiling this notebook's memory-aware
#     preprocessing (Section 5) sizes itself against. Falls back gracefully
#     to the raw core count / no RAM ceiling on an older project_config.json
#     that predates this block -- re-run 01_business_understanding.ipynb once
#     to pick it up (fast: it only reads label counts and writes config). ---
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)
MAX_RAM_BYTES = _resource_limits.get("max_ram_bytes")  # None on an older config -- handled below

FEATURE_ENG_DIR = PILLAR_DIRS["feature_engineering"]
MODEL_DEV_DIR = PILLAR_DIRS["model_development"]
MODEL_DEV_DIR.mkdir(parents=True, exist_ok=True)
MODELS_SUBDIR = MODEL_DEV_DIR / "models"
MODELS_SUBDIR.mkdir(parents=True, exist_ok=True)

TRAIN_SPLIT_ENG_PATH = Path(NB04_SUMMARY["output_files"]["train_split_engineered.csv"])
TEST_SPLIT_ENG_PATH = Path(NB04_SUMMARY["output_files"]["test_split_engineered.csv"])
TEST_ENGINEERED_PATH = Path(NB04_SUMMARY["output_files"]["test_engineered.parquet"])

for _p in (TRAIN_SPLIT_ENG_PATH, TEST_SPLIT_ENG_PATH, TEST_ENGINEERED_PATH):
    if not _p.exists():
        raise FileNotFoundError(f"Required file not found: {_p}")

print(f"Loaded config from     : {CONFIG_PATH}")
print(f"Loaded NB02 summary    : {NB02_SUMMARY_PATH}")
print(f"Loaded NB04 summary    : {NB04_SUMMARY_PATH}")
print(f"RANDOM_SEED             : {RANDOM_SEED} (same seed used by every notebook in this platform)")
print(f"DETECTED_LOGICAL_CORES  : {DETECTED_LOGICAL_CORES}")
print(f"WARP_THREAD_COUNT       : {WARP_THREAD_COUNT} (95% cap -- used for every model's n_jobs/thread_count below)")
print(f"MAX_RAM_BYTES           : "
      f"{f'{MAX_RAM_BYTES / 1e9:.1f} GB (90% cap)' if MAX_RAM_BYTES else 'not set -- re-run Notebook 01 to enable'}")
print(f"Model artifacts will be written under: {MODEL_DEV_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import gc

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    import joblib
except ImportError:
    missing.append("joblib")
try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
    from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
    from sklearn.inspection import permutation_importance
except ImportError:
    missing.append("scikit-learn")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )

try:
    from xgboost import XGBClassifier
    _HAS_XGBOOST = True
except ImportError:
    _HAS_XGBOOST = False

try:
    from lightgbm import LGBMClassifier
    _HAS_LIGHTGBM = True
except ImportError:
    _HAS_LIGHTGBM = False

try:
    from catboost import CatBoostClassifier
    _HAS_CATBOOST = True
except ImportError:
    _HAS_CATBOOST = False

logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4, Concurrency)")
print(f"xgboost available   : {_HAS_XGBOOST}")
print(f"lightgbm available  : {_HAS_LIGHTGBM}")
print(f"catboost available  : {_HAS_CATBOOST}" + ("" if _HAS_CATBOOST else "  (pip install catboost to include it)"))


def _rss_gb() -> float:
    """Current process resident memory, in GB -- printed at each major stage
    below so peak usage against the 90% RAM ceiling is visible as this
    notebook runs, not just inferred after the fact."""
    return psutil.Process().memory_info().rss / 1e9


_process_start_rss_gb = _rss_gb()
print(f"\nProcess RSS at Section 2 start: {_process_start_rss_gb:.2f} GB")
if MAX_RAM_BYTES:
    print(f"Configured RAM ceiling (90% of detected total): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: OFFICIAL AMEX COMPETITION METRIC -- IMPLEMENTED FROM SCRATCH
# =============================================================================
_section("SECTION 3: Official AMEX Competition Metric")


def amex_metric_numpy(y_true: "np.ndarray", y_pred: "np.ndarray") -> float:
    """Official American Express - Default Prediction competition metric:
    0.5 * (Normalized Weighted Gini) + 0.5 * (Top-4% Capture Rate).

    Non-defaulters (target=0) are weighted 20x relative to defaulters
    (target=1) in both sub-metrics -- this reflects the competition's
    real-world cost asymmetry between the two classes. This is a
    vectorized numpy equivalent of the official pandas reference
    implementation published by the competition host; it was
    cross-verified against that canonical implementation during this
    notebook's own build-time testing (a verification step, not part of
    this run's live output).
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    def top_four_percent_captured(yt, yp):
        order = np.argsort(-yp, kind="mergesort")
        yt_sorted = yt[order]
        weight = np.where(yt_sorted == 0, 20.0, 1.0)
        cum_weight = np.cumsum(weight)
        cutoff = 0.04 * weight.sum()
        mask = cum_weight <= cutoff
        total_pos = yt_sorted.sum()
        if total_pos == 0:
            return 0.0
        return float(yt_sorted[mask].sum() / total_pos)

    def weighted_gini(yt, yp):
        order = np.argsort(-yp, kind="mergesort")
        yt_sorted = yt[order]
        weight = np.where(yt_sorted == 0, 20.0, 1.0)
        random_cum = np.cumsum(weight / weight.sum())
        total_pos_weighted = (yt_sorted * weight).sum()
        if total_pos_weighted == 0:
            return 0.0
        cum_pos_found = np.cumsum(yt_sorted * weight)
        lorentz = cum_pos_found / total_pos_weighted
        return float(((lorentz - random_cum) * weight).sum())

    g_actual = weighted_gini(y_true, y_pred)
    g_perfect = weighted_gini(y_true, y_true)
    normalized_gini = g_actual / g_perfect if g_perfect != 0 else 0.0
    top4 = top_four_percent_captured(y_true, y_pred)
    return 0.5 * (normalized_gini + top4)


def top_four_percent_capture_only(y_true, y_pred) -> float:
    """Standalone top-4% capture rate (reported separately in the comparison
    table, in addition to being one half of amex_metric_numpy)."""
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    order = np.argsort(-y_pred, kind="mergesort")
    yt_sorted = y_true[order]
    weight = np.where(yt_sorted == 0, 20.0, 1.0)
    cum_weight = np.cumsum(weight)
    cutoff = 0.04 * weight.sum()
    mask = cum_weight <= cutoff
    total_pos = yt_sorted.sum()
    if total_pos == 0:
        return 0.0
    return float(yt_sorted[mask].sum() / total_pos)


print("amex_metric_numpy() defined -- 0.5 * Normalized Gini + 0.5 * Top-4% Capture Rate.")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LOAD ENGINEERED DATA & BUILD FEATURE MATRIX
# =============================================================================
_section("SECTION 4: Load Engineered Data & Build Feature Matrix")

# --- Read only the header first (via the stdlib csv module -- no Polars
#     inference involved) so the exact numeric/categorical column split can be
#     used to build an EXPLICIT schema_overrides dict before any data is read.
#     Letting Polars *infer* dtypes from a sample (the previous approach) is
#     fragile on a ~1,800-column, ~367K-row file: a column that happens to be
#     all-null in the sampled rows can get inferred as Utf8 (string) instead
#     of a numeric type, and a genuine "inf"/"-inf" token further down the
#     file then stays a literal string in that column rather than being
#     parsed as float infinity -- which is exactly what crashed
#     `.is_infinite()` on D_87_trend_delta. Forcing every column's dtype
#     explicitly at read time removes this failure mode entirely: Polars'
#     float parser recognizes "inf"/"-inf"/"nan" tokens natively when the
#     column is declared as a float dtype up front. ---
import csv as _csv
with open(TRAIN_SPLIT_ENG_PATH, "r", encoding="utf-8", newline="") as _f:
    _header = next(_csv.reader(_f))

if "target" not in _header:
    raise RuntimeError(f"Expected a 'target' column in {TRAIN_SPLIT_ENG_PATH.name} -- check Notebook 02/04 output.")

CATEGORICAL = ["B_30", "B_38", "D_63", "D_64", "D_66", "D_68",
               "D_114", "D_116", "D_117", "D_120", "D_126"]
categorical_encode_cols = [f"{c}_last" for c in CATEGORICAL if f"{c}_last" in _header]
non_feature_cols = {"customer_ID", "target"}
numeric_feature_cols = [c for c in _header if c not in non_feature_cols and c not in categorical_encode_cols]
all_feature_cols = numeric_feature_cols + categorical_encode_cols

SPLIT_CSV_SCHEMA = {"customer_ID": pl.Utf8, "target": pl.Int8}
for _c in categorical_encode_cols:
    SPLIT_CSV_SCHEMA[_c] = pl.Utf8
for _c in numeric_feature_cols:
    SPLIT_CSV_SCHEMA[_c] = pl.Float32

_t0 = time.time()
train_split_pl = pl.read_csv(str(TRAIN_SPLIT_ENG_PATH), schema_overrides=SPLIT_CSV_SCHEMA)
test_split_pl = pl.read_csv(str(TEST_SPLIT_ENG_PATH), schema_overrides=SPLIT_CSV_SCHEMA)
test_true_pl = pl.read_parquet(str(TEST_ENGINEERED_PATH))
print(f"Loaded train_split_engineered.csv: {train_split_pl.shape[0]:,} x {train_split_pl.shape[1]} "
      f"({time.time() - _t0:.1f}s)")
print(f"Loaded test_split_engineered.csv : {test_split_pl.shape[0]:,} x {test_split_pl.shape[1]} "
      f"(held-out, labeled, never trained on)")
print(f"Loaded test_engineered.parquet   : {test_true_pl.shape[0]:,} x {test_true_pl.shape[1]} "
      f"(true unlabeled AMEX test set -- submission target)")

if "target" not in train_split_pl.columns or "target" not in test_split_pl.columns:
    raise RuntimeError("Expected a 'target' column in both engineered split files -- check Notebook 02/04 output.")
if "target" in test_true_pl.columns:
    raise RuntimeError("test_engineered.parquet unexpectedly contains a 'target' column -- this must stay the "
                        "true unlabeled test set. Check Notebook 02/04 for an accidental label leak.")

# --- The feature count below (base NB02 aggregates x5 stats/numeric column +
#     x2/categorical column, plus NB04's trend/delta/ratio/interaction
#     features) is large by construction on the REAL dataset's ~179 numeric x
#     11 categorical raw columns -- this is expected multiplicative growth
#     from the aggregation + engineering pipeline, not an accidental
#     duplication. Notebook 04's own Section 3 output (numeric raw columns
#     eligible for trend/delta features) is the authoritative source for the
#     exact raw column count this run started from. ---
print(f"\nCategorical (label-encode) columns : {len(categorical_encode_cols)}")
print(f"Numeric feature columns             : {len(numeric_feature_cols)}")
print(f"Total feature columns               : {len(all_feature_cols)}")

_bytes_per_row_f32 = len(all_feature_cols) * 4
_total_rows_all_splits = train_split_pl.shape[0] + test_split_pl.shape[0] + test_true_pl.shape[0]
_projected_gb_f32 = _bytes_per_row_f32 * _total_rows_all_splits / 1e9
print(f"Projected combined float32 feature-matrix size (train+holdout+true-test): {_projected_gb_f32:.2f} GB")
if MAX_RAM_BYTES and _projected_gb_f32 * 1e9 > 0.5 * MAX_RAM_BYTES:
    print(f"Note: this is a large share of the {MAX_RAM_BYTES / 1e9:.1f} GB RAM ceiling -- Section 5 below uses "
          f"float32 (not float64), does all imputation/encoding in Polars rather than pandas, frees the source "
          f"Polars frames as soon as the numpy arrays are built, and scores the true test set in bounded chunks "
          f"to keep peak memory well under that ceiling.")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: PREPROCESSING -- LABEL ENCODING, IMPUTATION, INF-CLEANING (POLARS-NATIVE, FLOAT32)
# =============================================================================
_section("SECTION 5: Preprocessing (fit on train split only, memory-bounded)")

# --- Memory design (fixes the MemoryError this section used to raise on the
#     real ~925K-row unlabeled test set): everything below runs as native
#     Polars expressions -- vectorized, columnar, no Python-level per-value
#     loops and no pandas .replace([inf, -inf], nan), which builds a
#     full-frame boolean mask PER value being replaced and was the direct
#     cause of the crash on a frame this wide. All three frames are cast to
#     float32 (halves the raw footprint vs. float64) before any numpy arrays
#     are materialized, and each Polars source frame is deleted immediately
#     after its numpy array is extracted, with an explicit gc.collect() so
#     the freed memory is actually returned before the next big allocation. ---

_t0 = time.time()

# --- Inf/NaN -> null, then cast numeric feature columns to float32, for all
#     three frames. Checking is_nan() here as well as is_infinite() is
#     defense in depth: Notebooks 02 and 04 now clean inf -> null on the raw
#     statement values before their own std()/cov()/var() aggregations (the
#     original source of this project's inf-token issue), but a NaN can in
#     principle still be produced elsewhere in the pipeline -- and NaN is not
#     inf, so is_infinite() alone does not catch it. ---
_inf_clean_exprs = [
    pl.when(pl.col(c).is_infinite() | pl.col(c).is_nan()).then(None).otherwise(pl.col(c)).cast(pl.Float32).alias(c)
    for c in numeric_feature_cols
]
train_split_pl = train_split_pl.with_columns(_inf_clean_exprs)
test_split_pl = test_split_pl.with_columns(_inf_clean_exprs)
test_true_pl = test_true_pl.with_columns(_inf_clean_exprs)
print(f"Inf-cleaned and cast {len(numeric_feature_cols)} numeric columns to float32 in all 3 frames "
      f"({time.time() - _t0:.1f}s)")

# --- Categorical label-encoding: build the vocabulary from the TRAIN split
#     only, map every frame through the same {category: code} dict, with an
#     explicit -1 fallback for any category the test data has that train
#     never saw (replace_strict, not replace -- avoids the deprecated,
#     silently-defaulting form). ---
_t0 = time.time()
label_encoders = {}
for c in categorical_encode_cols:
    train_split_pl = train_split_pl.with_columns(pl.col(c).cast(pl.Utf8).fill_null("__missing__").alias(c))
    test_split_pl = test_split_pl.with_columns(pl.col(c).cast(pl.Utf8).fill_null("__missing__").alias(c))
    test_true_pl = test_true_pl.with_columns(pl.col(c).cast(pl.Utf8).fill_null("__missing__").alias(c))

    _cats = sorted(train_split_pl.get_column(c).unique().to_list())
    _mapping = {cat: i for i, cat in enumerate(_cats)}

    train_split_pl = train_split_pl.with_columns(
        pl.col(c).replace_strict(_mapping, default=-1, return_dtype=pl.Int32).alias(c))
    test_split_pl = test_split_pl.with_columns(
        pl.col(c).replace_strict(_mapping, default=-1, return_dtype=pl.Int32).alias(c))
    test_true_pl = test_true_pl.with_columns(
        pl.col(c).replace_strict(_mapping, default=-1, return_dtype=pl.Int32).alias(c))
    label_encoders[c] = {"classes": _cats}
print(f"Label-encoded {len(label_encoders)} categorical columns (fit on train only) in {time.time() - _t0:.1f}s")

# --- Median imputation: computed from the TRAIN split only, applied to all 3 ---
_t0 = time.time()
feature_medians = train_split_pl.select(
    [pl.col(c).median().fill_null(0.0).alias(c) for c in numeric_feature_cols]
).to_dicts()[0]
_impute_exprs = [pl.col(c).fill_null(feature_medians[c]) for c in numeric_feature_cols]
train_split_pl = train_split_pl.with_columns(_impute_exprs)
test_split_pl = test_split_pl.with_columns(_impute_exprs)
test_true_pl = test_true_pl.with_columns(_impute_exprs)
print(f"Imputed {len(feature_medians)} numeric columns with train-only medians in {time.time() - _t0:.1f}s")

# --- Materialize numpy arrays (float32) and immediately free the Polars source
#     frames -- do NOT hold both representations of the same data at once. ---
_t0 = time.time()
X_train = train_split_pl.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
y_train = train_split_pl.get_column("target").to_numpy().astype(np.int64, copy=False)
del train_split_pl
gc.collect()

X_holdout = test_split_pl.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
y_holdout = test_split_pl.get_column("target").to_numpy().astype(np.int64, copy=False)
del test_split_pl
gc.collect()

X_true_test = test_true_pl.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
true_test_customer_ids = test_true_pl.get_column("customer_ID").to_numpy()
del test_true_pl
gc.collect()
print(f"Materialized float32 numpy arrays and freed source Polars frames in {time.time() - _t0:.1f}s")

# --- Manual float32 standardization (train-only fit) for Logistic Regression --
#     avoids scikit-learn's StandardScaler, which upcasts to float64
#     internally and would silently double this array's memory back up. ---
_train_mean = X_train.mean(axis=0, dtype=np.float64).astype(np.float32)
_train_std = X_train.std(axis=0, dtype=np.float64).astype(np.float32)
_train_std[_train_std == 0] = 1.0  # guard constant columns against divide-by-zero

X_train_scaled = (X_train - _train_mean) / _train_std
X_holdout_scaled = (X_holdout - _train_mean) / _train_std
X_true_test_scaled = (X_true_test - _train_mean) / _train_std
scaler = {"mean": _train_mean, "std": _train_std}  # persisted in Section 12 for future scoring

print(f"\nX_train      : {X_train.shape}, dtype {X_train.dtype}, default rate {y_train.mean():.4%}, "
      f"{X_train.nbytes / 1e9:.2f} GB")
print(f"X_holdout    : {X_holdout.shape}, dtype {X_holdout.dtype}, default rate {y_holdout.mean():.4%} "
      f"(held-out, never trained on), {X_holdout.nbytes / 1e9:.2f} GB")
print(f"X_true_test  : {X_true_test.shape}, dtype {X_true_test.dtype} (unlabeled, submission target), "
      f"{X_true_test.nbytes / 1e9:.2f} GB")
print(f"Process RSS now: {_rss_gb():.2f} GB"
      + (f" (of {MAX_RAM_BYTES / 1e9:.1f} GB ceiling)" if MAX_RAM_BYTES else ""))
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: MODEL ZOO DEFINITION
# =============================================================================
_section("SECTION 6: Model Zoo Definition")


def build_model_zoo():
    zoo = {
        "logistic_regression": {
            "factory": lambda: LogisticRegression(
                max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED,
            ),
            "uses_scaled": True,
        },
        "random_forest": {
            "factory": lambda: RandomForestClassifier(
                n_estimators=300, max_depth=12, min_samples_leaf=20, class_weight="balanced_subsample",
                n_jobs=WARP_THREAD_COUNT, random_state=RANDOM_SEED,
            ),
            "uses_scaled": False,
        },
        "extra_trees": {
            "factory": lambda: ExtraTreesClassifier(
                n_estimators=300, max_depth=12, min_samples_leaf=20, class_weight="balanced_subsample",
                n_jobs=WARP_THREAD_COUNT, random_state=RANDOM_SEED,
            ),
            "uses_scaled": False,
        },
        "hist_gradient_boosting": {
            "factory": lambda: HistGradientBoostingClassifier(
                max_iter=300, max_depth=8, learning_rate=0.05, random_state=RANDOM_SEED,
            ),
            "uses_scaled": False,
        },
    }
    if _HAS_XGBOOST:
        zoo["xgboost"] = {
            "factory": lambda: XGBClassifier(
                n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                tree_method="hist", n_jobs=WARP_THREAD_COUNT, random_state=RANDOM_SEED,
                eval_metric="auc", verbosity=0,
            ),
            "uses_scaled": False,
        }
    if _HAS_LIGHTGBM:
        zoo["lightgbm"] = {
            "factory": lambda: LGBMClassifier(
                n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                n_jobs=WARP_THREAD_COUNT, random_state=RANDOM_SEED, verbosity=-1,
            ),
            "uses_scaled": False,
        }
    if _HAS_CATBOOST:
        zoo["catboost"] = {
            "factory": lambda: CatBoostClassifier(
                iterations=400, depth=6, learning_rate=0.05, thread_count=WARP_THREAD_COUNT,
                random_seed=RANDOM_SEED, verbose=False,
            ),
            "uses_scaled": False,
        }
    return zoo


MODEL_ZOO = build_model_zoo()
print(f"Model zoo contains {len(MODEL_ZOO)} models: {list(MODEL_ZOO.keys())}")
if len(MODEL_ZOO) < 7:
    print(f"Note: {7 - len(MODEL_ZOO)} of the intended 7 models were skipped (library not installed). "
          f"See Section 2 output above for which.")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: STRATIFIED 5-FOLD CROSS-VALIDATION (WITHIN THE TRAIN SPLIT)
# =============================================================================
_section("SECTION 7: Stratified 5-Fold Cross-Validation")

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

cv_results = {}
for name, spec in MODEL_ZOO.items():
    gc.collect()
    _t0 = time.time()
    fold_auc, fold_amex, fold_logloss = [], [], []
    Xc = X_train_scaled if spec["uses_scaled"] else X_train
    for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(Xc, y_train)):
        model = spec["factory"]()
        model.fit(Xc[tr_idx], y_train[tr_idx])
        proba = model.predict_proba(Xc[va_idx])[:, 1]
        fold_auc.append(roc_auc_score(y_train[va_idx], proba))
        fold_amex.append(amex_metric_numpy(y_train[va_idx], proba))
        fold_logloss.append(log_loss(y_train[va_idx], proba, labels=[0, 1]))
        del model
    cv_seconds = time.time() - _t0
    cv_results[name] = {
        "cv_auc_mean": float(np.mean(fold_auc)), "cv_auc_std": float(np.std(fold_auc)),
        "cv_amex_mean": float(np.mean(fold_amex)), "cv_amex_std": float(np.std(fold_amex)),
        "cv_logloss_mean": float(np.mean(fold_logloss)), "cv_seconds": round(cv_seconds, 1),
    }
    print(f"{name:<24} CV AUC {np.mean(fold_auc):.4f} +/- {np.std(fold_auc):.4f}   "
          f"CV AMEX {np.mean(fold_amex):.4f} +/- {np.std(fold_amex):.4f}   ({cv_seconds:.1f}s, {N_FOLDS} folds)")

print(f"\nProcess RSS after cross-validation: {_rss_gb():.2f} GB")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: FULL-TRAIN FIT + UNBIASED HOLDOUT EVALUATION
# =============================================================================
_section("SECTION 8: Full-Train Fit + Holdout Evaluation")

fitted_models = {}
holdout_results = {}
for name, spec in MODEL_ZOO.items():
    gc.collect()
    Xc_train = X_train_scaled if spec["uses_scaled"] else X_train
    Xc_holdout = X_holdout_scaled if spec["uses_scaled"] else X_holdout

    _t0 = time.time()
    model = spec["factory"]()
    model.fit(Xc_train, y_train)
    train_seconds = time.time() - _t0

    _t0 = time.time()
    proba = model.predict_proba(Xc_holdout)[:, 1]
    inference_seconds = time.time() - _t0

    holdout_auc = roc_auc_score(y_holdout, proba)
    holdout_amex = amex_metric_numpy(y_holdout, proba)
    holdout_top4 = top_four_percent_capture_only(y_holdout, proba)
    holdout_logloss = log_loss(y_holdout, proba, labels=[0, 1])
    holdout_brier = brier_score_loss(y_holdout, proba)

    fitted_models[name] = model
    holdout_results[name] = {
        "holdout_auc": float(holdout_auc), "holdout_amex_metric": float(holdout_amex),
        "holdout_top4pct_capture": float(holdout_top4), "holdout_logloss": float(holdout_logloss),
        "holdout_brier_score": float(holdout_brier), "train_seconds": round(train_seconds, 1),
        "inference_seconds_per_1k_rows": round(inference_seconds / max(len(y_holdout), 1) * 1000, 4),
    }
    print(f"{name:<24} Holdout AUC {holdout_auc:.4f}   Holdout AMEX {holdout_amex:.4f}   "
          f"Top-4% capture {holdout_top4:.4f}   (train {train_seconds:.1f}s)")

print(f"\nProcess RSS after full-train fits: {_rss_gb():.2f} GB")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: MODEL COMPARISON TABLE & CHAMPION SELECTION
# =============================================================================
_section("SECTION 9: Model Comparison Table & Champion Selection")

comparison_rows = []
for name in MODEL_ZOO:
    row = {"model": name}
    row.update(cv_results[name])
    row.update(holdout_results[name])
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).sort_values("holdout_amex_metric", ascending=False).reset_index(drop=True)
champion_name = comparison_df.iloc[0]["model"]
champion_model = fitted_models[champion_name]
champion_uses_scaled = MODEL_ZOO[champion_name]["uses_scaled"]

comparison_path = MODEL_DEV_DIR / "model_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print(comparison_df.to_string(index=False))
print(f"\nChampion model (highest holdout AMEX metric): {champion_name}")
print(f"  Holdout AUC          : {holdout_results[champion_name]['holdout_auc']:.4f}")
print(f"  Holdout AMEX metric  : {holdout_results[champion_name]['holdout_amex_metric']:.4f}")
print(f"  Holdout top-4% capture: {holdout_results[champion_name]['holdout_top4pct_capture']:.4f}")
print(f"\u2705 Saved -> {comparison_path}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: CHAMPION FEATURE IMPORTANCE
# =============================================================================
_section("SECTION 10: Champion Feature Importance")

if hasattr(champion_model, "feature_importances_"):
    importances = np.asarray(champion_model.feature_importances_, dtype=np.float64)
    importance_kind = "impurity/gain-based feature_importances_"
elif hasattr(champion_model, "coef_"):
    importances = np.abs(np.asarray(champion_model.coef_, dtype=np.float64)).ravel()
    importance_kind = "absolute standardized coefficient magnitude"
else:
    # Some models (e.g. sklearn's HistGradientBoostingClassifier) expose
    # neither feature_importances_ nor coef_. Fall back to permutation
    # importance on the held-out split -- model-agnostic, and still computed
    # live from this run's own champion model and holdout data, not assumed.
    _perm_X = X_holdout_scaled if champion_uses_scaled else X_holdout
    _t0 = time.time()
    _perm = permutation_importance(
        champion_model, _perm_X, y_holdout, scoring="roc_auc",
        n_repeats=5, random_state=RANDOM_SEED, n_jobs=WARP_THREAD_COUNT,
    )
    importances = np.asarray(_perm.importances_mean, dtype=np.float64)
    importance_kind = f"permutation importance on the holdout split (ROC AUC drop, 5 repeats, {time.time() - _t0:.1f}s)"

importance_df = pd.DataFrame({"feature": all_feature_cols, "importance": importances})
importance_df = importance_df.sort_values("importance", ascending=False).reset_index(drop=True)
importance_path = MODEL_DEV_DIR / "champion_feature_importance.csv"
importance_df.to_csv(importance_path, index=False)

print(f"Champion: {champion_name}  (importance basis: {importance_kind})")
print("Top 10 features:")
for _, r in importance_df.head(10).iterrows():
    print(f"  {r['feature']:<40} {r['importance']:.5f}")
print(f"\u2705 Saved -> {importance_path}")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: VALIDATED CHART STYLE (SAME SYSTEM AS NOTEBOOK 03)
# =============================================================================
_section("SECTION 11: Charts -- Model Comparison & Champion Feature Importance")

VIZ = {
    "surface": "#fcfcfb",
    "text_primary": "#0b0b0b",
    "text_secondary": "#52514e",
    "grid": "#e3e2dd",
    "cat_blue": "#2a78d6",
    "cat_red": "#e34948",
    "seq_blue_mid": "#3987e5",
    "diverging_neutral": "#f0efec",
}


def _style_axes(ax):
    ax.set_facecolor(VIZ["surface"])
    ax.figure.set_facecolor(VIZ["surface"])
    ax.grid(axis="y", color=VIZ["grid"], linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(VIZ["grid"])
    ax.tick_params(colors=VIZ["text_secondary"], labelsize=9)
    ax.title.set_color(VIZ["text_primary"])
    ax.xaxis.label.set_color(VIZ["text_secondary"])
    ax.yaxis.label.set_color(VIZ["text_secondary"])


# --- Chart 1: Model comparison -- holdout AUC vs. holdout AMEX metric, grouped bars ---
_models_sorted = comparison_df["model"].tolist()
_auc_vals = comparison_df["holdout_auc"].tolist()
_amex_vals = comparison_df["holdout_amex_metric"].tolist()
_x = np.arange(len(_models_sorted))
_width = 0.36

fig, ax = plt.subplots(figsize=(9, 5.5), dpi=150)
ax.bar(_x - _width / 2, _auc_vals, _width, label="Holdout AUC", color=VIZ["cat_blue"], zorder=3)
ax.bar(_x + _width / 2, _amex_vals, _width, label="Holdout AMEX Metric", color=VIZ["cat_red"], zorder=3)
_style_axes(ax)
ax.set_xticks(_x)
ax.set_xticklabels(_models_sorted, rotation=30, ha="right")
ax.set_ylabel("Score")
ax.set_title(f"Model Comparison on Held-Out Split (champion: {champion_name})")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
model_comparison_chart_path = MODEL_DEV_DIR / "model_comparison_chart.png"
fig.savefig(model_comparison_chart_path, dpi=150, facecolor=VIZ["surface"])
plt.show()
plt.close(fig)
print(f"\u2705 Saved -> {model_comparison_chart_path}")

# --- Chart 2: Champion feature importance, top 20 ---
_top20 = importance_df.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, max(4, 0.32 * len(_top20))), dpi=150)
ax.barh(_top20["feature"], _top20["importance"], color=VIZ["cat_blue"], zorder=3)
_style_axes(ax)
ax.grid(axis="x", color=VIZ["grid"], linewidth=0.8, zorder=0)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Importance")
ax.set_title(f"Top 20 Features -- Champion Model ({champion_name})")
fig.tight_layout()
feature_importance_chart_path = MODEL_DEV_DIR / "champion_feature_importance_chart.png"
fig.savefig(feature_importance_chart_path, dpi=150, facecolor=VIZ["surface"])
plt.show()
plt.close(fig)
print(f"\u2705 Saved -> {feature_importance_chart_path}")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: PERSIST ALL MODELS & PREPROCESSING ARTIFACTS
# =============================================================================
_section("SECTION 12: Persist Models & Preprocessing Artifacts")

for name, model in fitted_models.items():
    model_path = MODELS_SUBDIR / f"{name}.joblib"
    joblib.dump(model, model_path)
    print(f"\u2705 {model_path.name:<32} {model_path.stat().st_size / 1e6:>8,.2f} MB")

preprocessing_path = MODELS_SUBDIR / "preprocessing_artifacts.joblib"
joblib.dump({
    "label_encoders": label_encoders,
    "feature_medians": feature_medians,
    "scaler": scaler,
    "all_feature_cols": all_feature_cols,
    "categorical_encode_cols": categorical_encode_cols,
    "numeric_feature_cols": numeric_feature_cols,
}, preprocessing_path)
print(f"\u2705 {preprocessing_path.name:<32} {preprocessing_path.stat().st_size / 1e6:>8,.2f} MB")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: CHAMPION PREDICTIONS ON THE TRUE UNLABELED TEST SET -- SUBMISSION FILE
# =============================================================================
_section("SECTION 13: Champion Predictions -- Submission File")

# --- Scored in bounded chunks rather than one predict_proba() call on the
#     full ~925K-row array -- several model libraries allocate meaningful
#     temporary buffers during prediction that scale with input size, and
#     chunking keeps that transient cost flat and small regardless of how
#     large the true test set is. ---
CHUNK_SIZE = 100_000
Xc_true_test = X_true_test_scaled if champion_uses_scaled else X_true_test
n_true = Xc_true_test.shape[0]

_t0 = time.time()
true_test_proba = np.empty(n_true, dtype=np.float64)
for start in range(0, n_true, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, n_true)
    true_test_proba[start:end] = champion_model.predict_proba(Xc_true_test[start:end])[:, 1]
print(f"Scored {n_true:,} unlabeled test customers with champion '{champion_name}' in {time.time() - _t0:.1f}s "
      f"({CHUNK_SIZE:,}-row chunks, process RSS now {_rss_gb():.2f} GB)")

submission_df = pd.DataFrame({"customer_ID": true_test_customer_ids, "prediction": true_test_proba})
submission_path = MODEL_DEV_DIR / "submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"\u2705 Saved -> {submission_path} ({submission_df.shape[0]:,} rows)")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: VERIFICATION -- INTEGRITY CHECKS ON EVERYTHING THIS NOTEBOOK WROTE
# =============================================================================
_section("SECTION 14: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("submission.csv row count matches test_engineered.parquet row count",
       submission_df.shape[0] == n_true,
       f"({submission_df.shape[0]:,} vs {n_true:,})")
_check("submission.csv has no missing customer_ID values", submission_df["customer_ID"].isna().sum() == 0)
_check("submission.csv has no NaN predictions", submission_df["prediction"].isna().sum() == 0)
_check("submission.csv predictions are within [0, 1]",
       bool((submission_df["prediction"] >= 0).all() and (submission_df["prediction"] <= 1).all()))
_check("comparison_df contains one row per model in MODEL_ZOO", comparison_df.shape[0] == len(MODEL_ZOO))

_expected_files = [comparison_path, importance_path, model_comparison_chart_path,
                    feature_importance_chart_path, preprocessing_path, submission_path] + \
    [MODELS_SUBDIR / f"{n}.joblib" for n in fitted_models]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 05 verification checks failed. See \u274c lines above.")

print("\nAll Notebook 05 checks passed.")
print("\n\u2705 Section 14 complete.")


# =============================================================================
# SECTION 15: WRITE NOTEBOOK 05 SUMMARY ARTIFACT (for Notebook 17's rollup)
# =============================================================================
_section("SECTION 15: Write Notebook 05 Summary Artifact")

notebook_05_summary = {
    "notebook": "05_model_development",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "models_trained": list(MODEL_ZOO.keys()),
    "models_skipped_not_installed": [n for n in ["xgboost", "lightgbm", "catboost"]
                                      if n not in MODEL_ZOO],
    "resource_config_used": {
        "warp_thread_count": WARP_THREAD_COUNT,
        "max_ram_bytes": MAX_RAM_BYTES,
        "peak_process_rss_gb_observed": round(_rss_gb(), 2),
    },
    "n_cv_folds": N_FOLDS,
    "train_shape": list(X_train.shape),
    "holdout_shape": list(X_holdout.shape),
    "true_test_shape": list(X_true_test.shape),
    "champion_model": champion_name,
    "champion_metrics": {**cv_results[champion_name], **holdout_results[champion_name]},
    "full_comparison_table": comparison_rows,
    "output_files": {p.name: str(p) for p in _expected_files},
}
nb05_summary_path = ARTIFACTS_DIR / "notebook_05_summary.json"
with open(nb05_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_05_summary, f, indent=2)
print(f"\u2705 Saved -> {nb05_summary_path} (Notebook 17 reads this file to build the rolled-up Model "
      f"Development section)")
print("\n\u2705 Section 15 complete.")


# =============================================================================
# SECTION 16: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 16: Notebook 05 Complete -- Handoff to Notebook 06")

print("NOTEBOOK 05: MODEL DEVELOPMENT -- COMPLETE")
print(f"  Models trained & compared        : {len(MODEL_ZOO)} ({', '.join(MODEL_ZOO.keys())})")
print(f"  Champion (by holdout AMEX metric): {champion_name}")
print(f"  Champion holdout AUC             : {holdout_results[champion_name]['holdout_auc']:.4f}")
print(f"  Champion holdout AMEX metric     : {holdout_results[champion_name]['holdout_amex_metric']:.4f}")
print(f"  Champion holdout top-4% capture  : {holdout_results[champion_name]['holdout_top4pct_capture']:.4f}")
print(f"  Files produced                   : {len(_expected_files)}")
for _p in _expected_files:
    print(f"    - {_p.name}")
print(f"  Peak process RSS this run        : {_rss_gb():.2f} GB"
      + (f" (of {MAX_RAM_BYTES / 1e9:.1f} GB ceiling)" if MAX_RAM_BYTES else ""))
print(f"  Next notebook                    : 06_explainable_ai.ipynb (Sprint 2)")
print("\n\u2705 Ready to proceed.")
